# Aadhaar Enrolment Data — Data Loading and Cleaning

This notebook performs the initial data preparation and cleaning of the Aadhaar enrolment dataset using **PySpark**.

The main steps covered in this section are:

- Loading multiple CSV files containing Aadhaar enrolment records.
- Combining the datasets into a single DataFrame.
- Inspecting the schema, summary statistics, and row count.
- Converting columns to appropriate data types.
- Identifying and removing invalid state values.
- Standardising state and district names.
- Loading a reference dataset containing valid state, district, and pincode information.
- Identifying state names that do not match the reference dataset.
- Correcting known state-name inconsistencies.


In [1]:
from pyspark.sql.functions import col, to_date, col, lit, when, substring
from pyspark.sql.functions import lower, regexp_replace, trim
from pyspark.sql.types import StructType, StructField, StringType


StatementMeta(, 540234b1-b1ae-4652-99f6-300899dc6b27, 3, Finished, Available, Finished, False)

## 1. Load the Raw Aadhaar Enrolment Data

The Aadhaar enrolment data is distributed across three CSV files. Each file contains a portion of the overall dataset.

The files are loaded separately and will be combined into a single Spark DataFrame in the next step.

The `header` option is enabled so that the first row of each CSV file is interpreted as the column names.


In [2]:
# Load the three source CSV files into separate Spark DataFrames.
df1 = spark.read.format("csv")\
.option("header","true")\
.load("Files/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment_0_500000.csv")

df2 = spark.read.format("csv").option("header","true").load("Files/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment_1000000_1006029.csv")

df3 = spark.read.format("csv").option("header","true").load("Files/raw/api_data_aadhar_enrolment/api_data_aadhar_enrolment_500000_1000000.csv")

# Preview the first few records to confirm that the data was loaded correctly.
display(df1.limit(5))


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2cb334d0-fb54-43be-aea3-e92fe44613d3)

## 2. Combine the Enrolment Datasets

The three DataFrames contain different portions of the same dataset. They are combined using `union()` so that all records are stored in a single DataFrame.

The schemas of the source DataFrames are expected to be compatible because they originate from the same dataset.


In [3]:
enrol_df = df1.union(df2).union(df3)

# Display a sample of the combined DataFrame.
display(enrol_df.limit(5))


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a31fb890-56d5-4d5b-abb0-603918733e14)

## 3. Initial Data Inspection

Before performing transformations, the combined dataset is inspected to understand its structure and size.

The following checks are performed:

- **Schema:** Displays the column names and their current data types.
- **Descriptive statistics:** Provides summary statistics for applicable columns.
- **Row count:** Determines the total number of records in the combined dataset.


In [4]:
print(f"df schema : {enrol_df.printSchema()}")
print(f"df describe : {enrol_df.describe()}")
print(f"df rowCount : {enrol_df.count()}")

StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 6, Finished, Available, Finished, False)

root
 |-- date: string (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- pincode: string (nullable = true)
 |-- age_0_5: string (nullable = true)
 |-- age_5_17: string (nullable = true)
 |-- age_18_greater: string (nullable = true)

df schema : None
df describe : DataFrame[summary: string, date: string, state: string, district: string, pincode: string, age_0_5: string, age_5_17: string, age_18_greater: string]
df rowCount : 1006029


## 4. Data Type Conversion

The raw CSV data is initially read as strings. Appropriate data types are therefore assigned to the columns before further analysis.

The following transformations are applied:

- `date` is converted from a string into a date using the `dd-MM-yyyy` format.
- `state` and `district` are explicitly retained as strings.
- `pincode` is converted to an integer.
- The three age-group enrolment columns are converted to integers.

Using appropriate data types improves the reliability of subsequent filtering, aggregation, and analysis operations.


In [5]:
# Apply the required data type transformations to the enrolment dataset.
enrol_df = enrol_df.select(
    # Convert the date string into a Spark date type.
    to_date(col("date") , "dd-MM-yyyy").alias("date"),
    
    # Keep location fields as strings.
    col("state").cast("string"),
    col("district").cast("string"),
    
    # Convert pincode and age-group counts to integer values.
    col("pincode").cast("integer"),
    col("age_0_5").cast("integer"),
    col("age_5_17").cast("integer"),
    col("age_18_greater").cast("integer")
)

# Verify that the transformations produced the expected schema.
enrol_df.printSchema()


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 7, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- age_0_5: integer (nullable = true)
 |-- age_5_17: integer (nullable = true)
 |-- age_18_greater: integer (nullable = true)



## 5. Inspect State Values

The distinct values in the `state` column are examined to identify possible data-quality issues such as unexpected numeric values, inconsistent naming, or malformed entries.


In [6]:
enrol_df.select("state").distinct().show()

StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 8, Finished, Available, Finished, False)

+--------------------+
|               state|
+--------------------+
|            Nagaland|
|           Karnataka|
|              Odisha|
|              Kerala|
|         WEST BENGAL|
|Dadra and Nagar H...|
|              Ladakh|
|          Tamil Nadu|
|        Chhattisgarh|
|      Andhra Pradesh|
|         Lakshadweep|
|      Madhya Pradesh|
|              Punjab|
|             Manipur|
|         Daman & Diu|
|     Jammu & Kashmir|
|                 Goa|
|             Mizoram|
|Dadra and Nagar H...|
|    Himachal Pradesh|
+--------------------+
only showing top 20 rows



## 6. Remove Invalid State Values

During the inspection of the `state` column, the value `"100000"` was identified as an invalid state value.

The affected records are first inspected and counted. These records are then removed from the dataset so that invalid state values do not affect subsequent location-based analysis.


In [7]:
# Inspect records where the state contains the invalid value "100000".
enrol_df.filter(col("state") == "100000").show()

# Count the number of records containing the invalid state value.
count = enrol_df.filter(col("state") == "100000").count()
print(f"Total fields with incorrect data: {count}")

# Remove records containing the invalid state value.
enrol_df = enrol_df.filter(col("state") != "100000")


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 9, Finished, Available, Finished, False)

+----------+------+--------+-------+-------+--------+--------------+
|      date| state|district|pincode|age_0_5|age_5_17|age_18_greater|
+----------+------+--------+-------+-------+--------+--------------+
|2025-09-02|100000|  100000| 100000|      0|       0|             3|
|2025-09-03|100000|  100000| 100000|      0|       0|             1|
|2025-09-08|100000|  100000| 100000|      0|       0|             1|
|2025-09-09|100000|  100000| 100000|      0|       0|             1|
|2025-09-11|100000|  100000| 100000|      0|       0|             2|
|2025-09-12|100000|  100000| 100000|      0|       0|             2|
|2025-09-19|100000|  100000| 100000|      0|       0|             1|
|2025-09-20|100000|  100000| 100000|      0|       0|             1|
|2025-10-24|100000|  100000| 100000|      0|       1|             0|
|2025-11-15|100000|  100000| 100000|      0|       0|             3|
|2025-11-16|100000|  100000| 100000|      0|       0|             2|
|2025-11-17|100000|  100000| 10000

## 7. Standardise State and District Names

Location names may appear in different formats because of differences in capitalisation, punctuation, spacing, or spelling conventions.

A reusable `normalize_text()` function is created to standardise text values in the `state` and `district` columns.

The function:

1. Converts text to lowercase.
2. Replaces `&` with `and`.
3. Replaces spaces, dots, and hyphens with underscores.
4. Removes consecutive duplicate underscores.
5. Removes a leading `"the_"`.
6. Removes leading and trailing underscores or asterisks.
7. Trims unnecessary whitespace.

This creates a consistent format that makes it easier to compare location names with the reference dataset.


In [8]:
def normalize_text(col_obj):
    # Convert text to lowercase so that comparisons are case-insensitive.
    c = lower(col_obj)

    # Standardise the ampersand character.
    c = regexp_replace(c, "&", "and")

    # Replace spaces, dots, and hyphens with underscores.
    c = regexp_replace(c, r"[\s\.\-]+", "_")

    # Collapse consecutive underscores into a single underscore.
    c = regexp_replace(c, r"_+", "_")

    # Remove "the_" when it appears at the beginning of a value.
    c = regexp_replace(c, r"^the_", "")

    # Remove unwanted underscores and asterisks from the beginning and end.
    c = regexp_replace(c, r"^[_*]+", "")
    c = regexp_replace(c, r"[_*]+$", "")
    
    return trim(c)


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 10, Finished, Available, Finished, False)

## 8. Apply Text Standardisation

The normalisation function is applied to the `state` and `district` columns.

The original columns are temporarily renamed to `oldState` and `oldDistrict`. This preserves the original values while the cleaned versions are generated in the new `state` and `district` columns.

Keeping the original values temporarily available is useful for comparing the raw and cleaned data during validation.


In [9]:
enrol_df = enrol_df \
    .withColumnRenamed("state", "oldState") \
    .withColumnRenamed("district", "oldDistrict") \
    .withColumn("state", normalize_text(col("oldState"))) \
    .withColumn("district", normalize_text(col("oldDistrict")))

# Verify the resulting schema and inspect sample records.
enrol_df.printSchema()
display(enrol_df.limit(10))


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 11, Finished, Available, Finished, False)

root
 |-- date: date (nullable = true)
 |-- oldState: string (nullable = true)
 |-- oldDistrict: string (nullable = true)
 |-- pincode: integer (nullable = true)
 |-- age_0_5: integer (nullable = true)
 |-- age_5_17: integer (nullable = true)
 |-- age_18_greater: integer (nullable = true)
 |-- state: string (nullable = true)
 |-- district: string (nullable = true)



SynapseWidget(Synapse.DataFrame, 0e800990-c14a-4dd3-bd71-cc834b8c40f2)

## 9. Load the State, District and Pincode Reference Data

A separate reference dataset is loaded to validate the location information in the Aadhaar enrolment data.

The reference dataset contains:

- State names
- District names
- Pincodes

The same text-normalisation function used for the enrolment data is applied to the state and district names so that both datasets use a consistent naming format.

The `pincode` column is also converted to an integer to maintain compatible data types.


In [10]:
states_df = spark.read.format("csv").option("header","true").load("Files/raw/states/STATES.csv")

# Normalise location names and convert pincode to an integer.
states_df = states_df \
    .withColumn("state", normalize_text(col("statename"))) \
    .withColumn("district", normalize_text(col("district"))) \
    .withColumn("pincode", col("pincode").cast("integer")) \
    .select("state", "district", "pincode") \
    .filter(col("state").isNotNull() & (col("state") != "na"))

# Display the distinct state names available in the reference dataset.
display(states_df.select("state").distinct())


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 65753bc1-b86f-4591-9c09-7eed8cea3b5d)

## 10. Validate State Names Against the Reference Dataset

A `left_anti` join is used to identify state values that exist in the Aadhaar enrolment dataset but do not exist in the reference dataset.

This is useful for detecting naming inconsistencies or values that require further standardisation.

Only distinct unmatched state names are displayed.


In [11]:
missing_states = enrol_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

display(missing_states)


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 12b4c88f-9f91-4e73-84ce-6815209f91eb)

## 11. Correct Known State Name Variations

The unmatched state values are reviewed and known naming variations are standardised to the corresponding reference-dataset names.

The following mappings are applied:

- `pondicherry` → `puducherry`
- `orissa` → `odisha`
- `westbengal` and `west_bangal` → `west_bengal`
- `dadra_and_nagar_haveli` and `daman_and_diu` → `dadra_and_nagar_haveli_and_daman_and_diu`

Values that do not match any of these known variations are retained using `otherwise()`.


In [12]:
from pyspark.sql import functions as F

enrol_df = enrol_df.withColumn(
    "state", 
    F.when(F.col("state") == "pondicherry", "puducherry")
     .when(F.col("state") == "orissa", "odisha")
     .when(F.col("state").isin("westbengal", "west_bangal"), "west_bengal")
     .when(F.col("state").isin("dadra_and_nagar_haveli", "daman_and_diu"), "dadra_and_nagar_haveli_and_daman_and_diu")
     .otherwise(F.col("state"))
)


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 14, Finished, Available, Finished, False)

## 12. Validate the State Corrections

The state validation is repeated after applying the known corrections.

If the cleaning has successfully resolved the identified naming inconsistencies, the number of unmatched states should decrease.

Any remaining values require further investigation before proceeding with downstream analysis.


In [13]:
missing_states = enrol_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

missing_states.show()


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 15, Finished, Available, Finished, False)

+-----+
|state|
+-----+
+-----+



## 13. Validate District Names Against the Reference Dataset

After validating the state values, the district values are checked against the trusted `states_df` reference dataset.

A `left_anti` join identifies district values present in the enrolment dataset that cannot be matched to the reference dataset.

The number of distinct unmatched districts is calculated to quantify the remaining data-quality issues.


In [14]:
# Validate the state-district combination against the trusted reference data.
missing_districts = enrol_df.join(
    states_df.select("state", "district").distinct(),
    on=["state", "district"],
    how="left_anti"
).select("state", "district").distinct()

print(f"Number of unmatched state-district combinations: {missing_districts.count()}")


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 16, Finished, Available, Finished, False)

Number of unmatched state-district combinations: 254


## 14. Correct Invalid District and State Values Using Pincode

The reference dataset is treated as the authoritative source for location information.

When an enrolment record contains an unmatched `state`-`district` combination, its `pincode` is used to look up the corresponding valid state and district from `states_df`.

The correction process follows these steps:

1. Identify records with an invalid state-district combination.
2. Create a pincode-based lookup table from the trusted reference dataset.
3. Remove duplicate pincodes from the lookup to avoid unnecessary row multiplication during the join.
4. Join the lookup information to the enrolment dataset.
5. Replace the state and district only when:
   - the original state-district combination is invalid, and
   - a valid reference match is available for the pincode.
6. Remove the temporary columns used during the correction process.

This approach allows the pincode to act as the reference key for correcting location information.


In [15]:
import pyspark.sql.functions as F

# Identify records whose state-district combination does not exist
# in the trusted reference dataset.
invalid_location_df = enrol_df.join(
    states_df.select("state", "district").distinct(),
    on=["state", "district"],
    how="left_anti"
).select(
    "state",
    "district"
).distinct() \
 .withColumn("is_invalid_location", F.lit(True))


# Create a pincode-based reference lookup containing the trusted
# state and district values.
#
# dropDuplicates() prevents duplicate reference records from
# multiplying rows when the lookup is joined to the enrolment data.
pincode_lookup_df = states_df.select(
    "pincode",
    F.col("state").alias("correct_state"),
    F.col("district").alias("correct_district")
).dropDuplicates(["pincode"])


# Attach the invalid-location flag and the reference location
# associated with each pincode.
updated_df = enrol_df.join(
    invalid_location_df,
    on=["state", "district"],
    how="left"
).join(
    pincode_lookup_df,
    on="pincode",
    how="left"
)


# Replace the state and district only when the original
# state-district combination is invalid and a valid pincode
# reference value is available.
enrol_df = updated_df.withColumn(
    "district",
    F.when(
        F.col("is_invalid_location").isNotNull() &
        F.col("correct_district").isNotNull(),
        F.col("correct_district")
    ).otherwise(F.col("district"))
).withColumn(
    "state",
    F.when(
        F.col("is_invalid_location").isNotNull() &
        F.col("correct_state").isNotNull(),
        F.col("correct_state")
    ).otherwise(F.col("state"))
).drop(
    "is_invalid_location",
    "correct_state",
    "correct_district"
)


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 17, Finished, Available, Finished, False)

In [16]:
district_mapping = {
    "andhra_pradesh": {
        "khammam": {"telangana": "khammam"},
        "nalgonda": {"telangana": "nalgonda"},
        "sri_potti_sriramulu_nellore": {"andhra_pradesh": "spsr_nellore"},
        "nellore": {"andhra_pradesh": "spsr_nellore"},
        "hyderabad": {"telangana": "hyderabad"},
        "k_v_rangareddy": {"telangana": "ranga_reddy"},
        "rangareddi": {"telangana": "ranga_reddy"},
        "mahabubnagar": {"telangana" : "mahabubnagar"},
        "mahbubnagar": {"telangana" : "mahabubnagar"},
        "visakhapatnam" : {"andhra_pradesh":"visakhapatanam"}
    },
    "bihar": {
        "west_champaran": {"bihar": "pashchim_champaran"},
        "east_champaran": {"bihar": "purbi_champaran"},
        "purba_champaran": {"bihar": "purbi_champaran"},
        "purnea" : {"bihar": "purnia"},
        "samstipur" :{"bihar":"samastipur"},
        "monghyr" : {"bihar" : "munger"},
        "aurangabad(bh)" : {"bihar" : "aurangabad"}
    },
    "chhattisgarh": {
        "dakshin_bastar_dantewada": {"chhattisgarh": "dantewada"},
    },
    "gujarat": {
        "ahmedabad": {"gujarat": "ahmadabad"},
        "panchmahals": {"gujarat": "panch_mahals"},
        "surendra_nagar": {"gujarat": "surendranagar"},
    },
    "haryana": {
        "yamuna_nagar": {"haryana": "yamunanagar"},
    },
    "jammu_and_kashmir": {
        "punch": {"jammu_and_kashmir": "poonch"},
        "baramula": {"jammu_and_kashmir": "baramulla"},
        "kargil": {"ladakh": "kargil"},
        "leh": {"ladakh": "leh_ladakh"}
    },
    "jharkhand": {
        "purbi_singhbhum": {"jharkhand": "east_singhbum"},
        "pashchimi_singhbhum": {"jharkhand": "west_singhbhum"},
        "east_singhbhum": {"jharkhand": "east_singhbum"},
        "seraikela_kharsawan":{"jharkhand" : "saraikela_kharsawan"}
    },
    "karnataka": {
        "chickmagalur": {"karnataka": "chikkamagaluru"},
        "hasan": {"karnataka": "hassan"},
        "bijapur": {"karnataka": "vijayapura"},
        "shimoga": {"karnataka": "shivamogga"},
        "mysore": {"karnataka": "mysuru"},
        "belgaum": {"karnataka": "belagavi"},
        "tumkur": {"karnataka": "tumakuru"},
        "ramanagar": {"karnataka": "ramanagara"},
        "chikmagalur" : {"karnataka":"chikkamagaluru"},
        "bangalore_rural" : {"karnataka":"bengaluru_rural"}
    },
    "ladakh": {
        "leh": {"ladakh": "leh_ladakh"},
    },
    "madhya_pradesh": {
        "narsimhapur": {"madhya_pradesh": "narsinghpur"},
        "ashok_nagar" : {"madhya_pradesh":"ashoknagar"}
    },
    "maharashtra": {
        "ahmadnagar": {"maharashtra": "ahmednagar"},
        "mumbai(_sub_urban_)": {"maharashtra": "mumbai_suburban"},
        "mumbai(_sub_urban_)": {"maharashtra": "mumbai_suburban"},
        "mumbai_city": {"maharashtra": "mumbai"},
        "ahilyanagar" : {"maharashtra":"ahmednagar"},
        "ahmed_nagar" : {"maharashtra" : "ahmednagar"}
    },
    "mizoram": {
        "mammit": {"mizoram": "mamit"},
    },
    "odisha": {
        "angul": {"odisha": "anugul"},
        "subarnapur": {"odisha": "sonepur"},
        "baleswar": {"odisha": "baleshwar"},
        "balasore" : {"odisha": "baleshwar"},
        "jagatsinghpur" : {"odisha":"jagatsinghapur"}
    },
    "puducherry": {
        "puducherry": {"puducherry": "pondicherry"},
    },
    "punjab": {
        "sas_nagar_(mohali)": {"punjab": "s_a_s_nagar"},
        "sas_nagar" : {"punjab": "s_a_s_nagar"},
        "firozpur" : {"punjab":"firozepur"},
        "shaheed_bhagat_singh_nagar" :{"punjab" : "shahid_bhagat_singh_nagar"}
    },
    "rajasthan": {
        "jhunjhunun": {"rajasthan": "jhunjhunu"},
        "chittaurgarh": {"rajasthan": "chittorgarh"},
        "didwana_kuchaman": {"maharashtra":"nagpur"},
        "khairthal_tijara":{"rajasthan" : "alwar"}
    },
    "sikkim": {
        "east_sikkim": {"sikkim": "east_district"},
        "east": {"sikkim": "east_district"},
        "sikkim": {"sikkim": "east_district"},
        "north_sikkim": {"sikkim": "north_district"},
        "north": {"sikkim": "north_district"},
        "gangtok" : {"sikkim" : "east_district"},
        "mangan" : {"sikkim" : "north_district"}
    },
    "tamil_nadu": {
        "kancheepuram": {"tamil_nadu": "kanchipuram"},
        "kanyakumari": {"tamil_nadu": "kanniyakumari"},
        "tiruvallur": {"tamil_nadu": "thiruvallur"},
        "thoothukkudi" : {"tamil_nadu":"tuticorin"}
    },
    "telangana": {
        "k_v_rangareddy": {"telangana": "ranga_reddy"},
        "rangareddy": {"telangana": "ranga_reddy"},
    },
    "uttar_pradesh": {
        "allahabad": {"uttar_pradesh": "prayagraj"},
        "bara_banki": {"uttar_pradesh": "barabanki"},
        "bulandshahar": {"uttar_pradesh": "bulandshahr"},
        "sant_kabir_nagar": {"uttar_pradesh": "sant_kabeer_nagar"},
        "faizabad": {"uttar_pradesh": "ayodhya"},
        "sant_ravidas_nagar" : {"uttar_pradesh":"bhadohi"},
        "bagpat" : {"uttar_pradesh":"baghpat"}
    },
    "uttarakhand": {
        "udham_singh_nagar": {"uttarakhand": "udam_singh_nagar"},
        "garhwal" : {"uttarakhand" : "tehri_garhwal"}
    },
    "west_bengal": {
        "dakshin_dinajpur" : {"west_bengal": "dinajpur_dakshin"},
        "paschim_medinipur": {"west_bengal": "medinipur_west"},
        "north_dinajpur": {"west_bengal": "dinajpur_uttar"},
        "bardhaman": {"west_bengal": "purba_bardhaman"},
        "malda": {"west_bengal": "maldah"},
        "north_24_parganas": {"west_bengal": "24_paraganas_north"},
        "south_twenty_four_parganas": {"west_bengal": "24_paraganas_south"},
        "puruliya": {"west_bengal": "purulia"},
        "cooch_behar": {"west_bengal": "coochbehar"},
        "purba_medinipur": {"west_bengal": "medinipur_east"},
        "koch_bihar": {"west_bengal": "coochbehar"},
        "haora": {"west_bengal": "howrah"},
        "dakshin_dinajpur": {"west_bengal": "dinajpur_dakshin"},
        "barddhaman": {"west_bengal": "purba_bardhaman"},
        "south_24_parganas": {"west_bengal": "24_paraganas_south"},
        "uttar_dinajpur": {"west_bengal": "dinajpur_uttar"},
        "south_dinajpur": {"west_bengal": "dinajpur_dakshin"},
        "medinipur": {"west_bengal": "medinipur_west"},
        "north_twenty_four_parganas": {"west_bengal": "24_paraganas_south"},
        "east_midnapore": {"west_bengal": "medinipur_east"},
        "darjiling": {"west_bengal": "darjeeling"},
        "west_midnapore": {"west_bengal": "medinipur_west"},
        "hugli": {"west_bengal": "hooghly"},
        
    },
}



from pyspark.sql import functions as F

# Build mapping:
# "source_state\tsource_district" -> "target_state\ttarget_district"
mapping_expr = F.create_map(
    *[
        F.lit(item)
        for state, districts in district_mapping.items()
        for district, target in districts.items()
        for target_state, target_district in target.items()
        for item in (
            f"{state}\t{district}",
            f"{target_state}\t{target_district}"
        )
    ]
)

# Create source state-district key and map it to the target state-district
enrol_df = (
    enrol_df
    .withColumn(
        "_state_district_key",
        F.concat_ws("\t", F.col("state"), F.col("district"))
    )
    .withColumn(
        "_mapped_state_district",
        mapping_expr[F.col("_state_district_key")]
    )
    .withColumn(
        "state",
        F.coalesce(
            F.split(F.col("_mapped_state_district"), "\t").getItem(0),
            F.col("state")
        )
    )
    .withColumn(
        "district",
        F.coalesce(
            F.split(F.col("_mapped_state_district"), "\t").getItem(1),
            F.col("district")
        )
    )
    .drop("_state_district_key", "_mapped_state_district")
)

# Re-check state-district combinations against the trusted reference dataset.
missing_districts = (
    enrol_df
    .join(
        states_df.select("state", "district").distinct(),
        on=["state", "district"],
        how="left_anti"
    )
    .select("state", "district")
    .distinct()
)

display(missing_districts)


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e29e42dd-8be1-4798-ab37-a6cdc8a15764)

In [17]:
bengaluru_df = enrol_df.filter(enrol_df["district"] == "bengaluru") \
              .select("state", "district", "pincode") \
              .distinct()

# Show the results
bengaluru_df.show()

StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 19, Finished, Available, Finished, False)

+---------+---------+-------+
|    state| district|pincode|
+---------+---------+-------+
|karnataka|bengaluru| 560039|
|karnataka|bengaluru| 560019|
|karnataka|bengaluru| 560069|
|karnataka|bengaluru| 560028|
|karnataka|bengaluru| 560046|
|karnataka|bengaluru| 560052|
|karnataka|bengaluru| 560014|
+---------+---------+-------+



In [18]:
# map bengaluru district
enrol_df = enrol_df.withColumn(
    "district",
    when(substring(col("pincode").cast("string"), 1, 3) == "560", "bengaluru_urban")
    .when(substring(col("pincode").cast("string"), 1, 3).isin("561","571", "562"), "bengaluru_rural")
    .otherwise(col("district"))
)

StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 20, Finished, Available, Finished, False)

In [19]:
missing_states = enrol_df.join(
    states_df, 
    on="state", 
    how="left_anti"
).select("state").distinct()

missing_states.show()


# Re-check state-district combinations against the trusted reference dataset.
missing_districts = (
    enrol_df
    .join(
        states_df.select("state", "district").distinct(),
        on=["state", "district"],
        how="left_anti"
    )
    .select("state", "district")
    .distinct()
)

missing_districts.show()


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 21, Finished, Available, Finished, False)

+-----+
|state|
+-----+
+-----+

+-----+--------+
|state|district|
+-----+--------+
+-----+--------+



In [21]:

enrol_df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("SILVER_Enriched_LAKEHOUSE.Enrolment")


StatementMeta(, d00d62d4-1eb2-409e-b024-b2e572a909cc, 24, Finished, Available, Finished, False)